In [2]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box

Create a dataframe with all spots that are active for at least six months with at at most 14 days between any two observations

In [43]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
df["date"] = pd.to_datetime(df["created_at_local"].dt.date)

# ignore standing water type and stream type categories because they do not require time series
df = df[~df["Category"].isin(["standing water type", "stream type"])]

# dates per spot and category
spot_dates = (
    df
    .groupby(["longitude", "latitude", "Category", "date"])
    .size()
    .reset_index(name="n_obs")
)

def get_all_streaks(dates):
    dates = sorted(set(dates))
    streaks = []
    start = dates[0]
    prev = dates[0]

    for d in dates[1:]:
        gap = (pd.Timestamp(d) - pd.Timestamp(prev)).days
        if gap <= 14:
            prev = d
        else:
            duration = (pd.Timestamp(prev) - pd.Timestamp(start)).days
            if duration >= 180: # about half a year
                streaks.append({
                    "streak_start": start,
                    "streak_end": prev,
                    "streak_days": duration
                })
            start = d
            prev = d

    # last streak
    duration = (pd.Timestamp(prev) - pd.Timestamp(start)).days
    if duration >= 180:
        streaks.append({
            "streak_start": start,
            "streak_end": prev,
            "streak_days": duration
        })

    return pd.DataFrame(streaks)

streak_periods = (
    spot_dates
    .groupby(["longitude", "latitude", "Category"])["date"]
    .apply(get_all_streaks)
    .reset_index(level=3, drop=True)
    .reset_index()
)

persistent = streak_periods.copy().reset_index(drop=True)
persistent["streak_start"] = pd.to_datetime(persistent["streak_start"])
persistent["streak_end"] = pd.to_datetime(persistent["streak_end"])

# country
spot_country = (
    df
    .groupby(["longitude", "latitude"])["Country"]
    .first()
    .reset_index()
)
persistent = persistent.merge(spot_country, on=["longitude", "latitude"], how="left")

# users active during the streak at this spot
df_merged = df.merge(
    persistent[["longitude", "latitude", "Category", "streak_start", "streak_end"]],
    on=["longitude", "latitude", "Category"],
    how="inner"
)

df_merged = df_merged[
    (df_merged["date"] >= df_merged["streak_start"]) &
    (df_merged["date"] <= df_merged["streak_end"])
]

spot_users = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
    .nunique()
    .reset_index(name="n_users")
)
persistent = persistent.merge(spot_users, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

spot_user_ids = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
    .apply(lambda x: [int(i) for i in x.unique()])
    .reset_index(name="user_ids")
)
persistent = persistent.merge(spot_user_ids, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

spot_obs = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])
    .size()
    .reset_index(name="n_obs")
)
persistent = persistent.merge(spot_obs, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

# final table
persistent = persistent[["latitude", "longitude", "Country", "streak_days", "streak_start", "streak_end", "n_users", "user_ids", "n_obs", "Category"]]
persistent = persistent.sort_values("streak_days", ascending=False).reset_index(drop=True)

print(persistent.head(20))
print(f"number of spots with >=180 days consecutive activity: {len(persistent)}")

persistent.to_csv("../Products/CSVs/persistent_spots_14d.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_12332\2540333354.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

     latitude   longitude      Country  streak_days streak_start streak_end  \
0   49.375062    8.889207      Germany       1822.0   2020-07-28 2025-07-24   
1   49.375012    8.888853      Germany       1822.0   2020-07-28 2025-07-24   
2   51.060439 -115.328485       Canada       1763.0   2019-08-02 2024-05-30   
3   47.726072   13.065014      Austria       1410.0   2017-12-25 2021-11-04   
4   47.395523    8.730670  Switzerland        791.0   2022-07-11 2024-09-09   
5   47.394939    8.733537  Switzerland        791.0   2022-07-11 2024-09-09   
6   47.394939    8.733537  Switzerland        761.0   2020-05-22 2022-06-22   
7   47.789612   13.068625      Austria        755.0   2018-08-17 2020-09-10   
8   48.328912   16.214674      Austria        711.0   2020-07-28 2022-07-09   
9   47.387870    8.563581  Switzerland        642.0   2020-11-03 2022-08-07   
10  47.391386    8.560455  Switzerland        642.0   2020-11-03 2022-08-07   
11  47.389403    8.561560  Switzerland        642.0 

And the same for spots that were active at least once a month for at least 12 months consecutively

In [44]:
df = pd.read_csv("../CWData_clean7.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")

# ignore standing water type and stream type categories because they do not require time series
df = df[~df["Category"].isin(["standing water type", "stream type"])]

# months per spot
spot_month = (
    df
    .groupby(["longitude", "latitude", "Category", "year_month"])
    .size()
    .reset_index(name="n_obs")
)

def get_all_streaks(months):
    months = sorted(set(months))
    streaks = []
    start = months[0]
    count = 1

    for i in range(1, len(months)):
        if months[i] == months[i-1] + 1:
            count += 1
        else:
            if count >= 12:
                streaks.append({
                    "streak_start": start,
                    "streak_end": months[i-1],
                    "max_streak": count
                })
            start = months[i]
            count = 1

    # letzten Streak nicht vergessen
    if count >= 12:
        streaks.append({
            "streak_start": start,
            "streak_end": months[-1],
            "max_streak": count
        })

    return pd.DataFrame(streaks)

streak_periods = (
    spot_month
    .groupby(["longitude", "latitude", "Category"])["year_month"]
    .apply(get_all_streaks)
    .reset_index(level=3, drop=True)
    .reset_index()
)

# only keep months with 12+ consecutively active months
persistent = streak_periods[streak_periods["max_streak"] >= 12].copy().reset_index(drop=True)

# country
spot_country = (
    df
    .groupby(["longitude", "latitude"])["Country"]
    .first()
    .reset_index()
)
persistent = persistent.merge(spot_country, on=["longitude", "latitude"], how="left")

# users active during the streak at this spot
df_merged = df.merge(
    persistent[["longitude", "latitude", "Category", "streak_start", "streak_end"]],
    on=["longitude", "latitude", "Category"],
    how="inner"
)

df_merged = df_merged[
    (df_merged["year_month"] >= df_merged["streak_start"]) &
    (df_merged["year_month"] <= df_merged["streak_end"])
]

spot_users = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
    .nunique()
    .reset_index(name="n_users")
)
persistent = persistent.merge(spot_users, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

spot_user_ids = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
    .apply(lambda x: [int(i) for i in x.unique()])
    .reset_index(name="user_ids")
)
persistent = persistent.merge(spot_user_ids, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

spot_obs = (
    df_merged
    .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])
    .size()
    .reset_index(name="n_obs")
)
persistent = persistent.merge(spot_obs, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

# final table
persistent = persistent[["latitude", "longitude", "Country", "max_streak", "streak_start", "streak_end", "n_users", "user_ids", "n_obs", "Category"]]
persistent = persistent.sort_values("max_streak", ascending=False).reset_index(drop=True)

print(persistent.head(20))
print(f"number of spots with >=12 months consecutive activity: {len(persistent)}")

persistent.to_csv("../Products/CSVs/persistent_spots_monthly.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_12332\3582902744.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

     latitude   longitude         Country  max_streak streak_start streak_end  \
0   47.789612   13.068625         Austria        74.0      2018-08    2024-09   
1   51.608782   -0.881946  United Kingdom        70.0      2020-04    2026-01   
2   51.580956   -0.873880  United Kingdom        70.0      2020-04    2026-01   
3   51.581133   -0.873914  United Kingdom        70.0      2020-04    2026-01   
4   51.602487   -0.880738  United Kingdom        70.0      2020-04    2026-01   
5   51.562064   -0.867595  United Kingdom        70.0      2020-04    2026-01   
6   51.574168   -0.871882  United Kingdom        70.0      2020-04    2026-01   
7   51.602152   -0.880616  United Kingdom        70.0      2020-04    2026-01   
8   51.571622   -0.870799  United Kingdom        69.0      2020-04    2025-12   
9   47.394939    8.733537     Switzerland        69.0      2020-05    2026-01   
10  51.060439 -115.328485          Canada        63.0      2019-04    2024-06   
11  52.193365    9.082108   

Create a maps of these two persistent spot criteria.

In [67]:
def persistent_mapper(filename):
    persistent = pd.read_csv(f"../Products/CSVs/{filename}.csv")
    world = gpd.read_file("../ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")

    category_colors = {
        "physical scale": "#1418fc",
        "plastic pollution": "#fbff2b",
        "soil moisture": "#82571b",
        "standing water type": "#dd6ef0",
        "stream type": "#fc1471",
        "temporary stream": "#8c14fc",
        "virtual scale": "#14c2fc"
    }

    gdf = gpd.GeoDataFrame(
        persistent,
        geometry=gpd.points_from_xy(persistent["longitude"], persistent["latitude"]),
        crs="EPSG:4326"
    ).to_crs("+proj=robin")
    gdf = gdf.sort_values("n_users", ascending=False)

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))

    world.to_crs("+proj=robin").plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )

    for cat, color in category_colors.items():
        subset = gdf[gdf["Category"] == cat]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset.geometry.x,
            subset.geometry.y,
            s=np.log1p(subset["n_users"]) * 70,
            color=color,
            edgecolor="black",
            linewidth=0.3,
            alpha=0.8,
            label=cat,
            zorder=2
        )

    # legend categories
    legend_cat_handles = [
        plt.scatter([], [], s=50, color=color, edgecolor="black", linewidth=0.3, alpha=0.8, label=cat.title())
        for cat, color in category_colors.items()
        if cat in gdf["Category"].unique()
    ]
    legend_cats = ax.legend(
        handles=legend_cat_handles,
        title="Category",
        loc="lower left",
        bbox_to_anchor=(0, 0.18),
        frameon=True,
        title_fontsize=12
    )
    legend_cats.get_title().set_fontsize(12)

    # legend user counts
    user_counts = [1, 5, 10, 20]
    legend_handles = [
        plt.scatter([], [], s=np.log1p(u) * 70, color="lightgrey",
                    edgecolor="black", linewidth=0.3, alpha=0.8, label=str(u))
        for u in user_counts
    ]
    legend_users = ax.legend(
        handles=legend_handles,
        title="Number of Unique Users",
        title_fontsize=12,
        loc="lower left",
        frameon=True,
    )
    legend_users.get_title().set_fontsize(12)
    ax.add_artist(legend_cats)  # show both legends

    if "14d" in filename:
        ax.set_title("Persistent Spots (Every 14 Days for ≥180 Days)", fontsize=15)
        plt.savefig("../Products/persistent_spots_map_14d.png", dpi=300, bbox_inches="tight")
    else:
        ax.set_title("Persistent Spots (Every Month for ≥12 Consecutive Months)", fontsize=15)
        plt.savefig("../Products/persistent_spots_map_monthly.png", dpi=300, bbox_inches="tight")
    ax.set_axis_off()
    plt.close()

In [68]:
for file in ["persistent_spots_monthly", "persistent_spots_14d"]:
    persistent_mapper(file)